# Validación de paneles y dimensiones

Este notebook ejecuta un conjunto exhaustivo de aserciones sobre todos los paneles largos y dimensiones para garantizar que la base es apta para regresión. Se corre después de regenerar cualquier panel.

**Si una sola aserción falla, el notebook se interrumpe**. La salida final imprime un resumen consolidado por bloque.

Bloques de validación:

1. **Esquema** — todos los archivos esperados existen, tienen las columnas esperadas, tipos correctos.
2. **Integridad de claves** — unicidad, no-nulls en columnas críticas.
3. **Cobertura temporal** — rangos de fechas dentro de lo esperado.
4. **Cobertura de entidades** — las entidades activas aparecen en los paneles, las inactivas también si tuvieron actividad pre-baja.
5. **Cuentas y dimensiones** — todos los códigos en los paneles existen en las dimensiones; no hay códigos huérfanos.
6. **Spot checks de eventos conocidos** — las cuentas CERA tienen saldo > 0 a partir de jul-2024; el panel pre-2020 captura el blanqueo Macri (2016-17).
7. **Consistencia agregado vs individuales** — la suma de los bancos individuales coincide con los agregados AAxxx (con tolerancia documentada).
8. **Crosswalk cuenta_categoria** — todas las cuentas del crosswalk existen en `dim_cuentas`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, EXTERNAL, REPO

import pandas as pd
import duckdb

CW_CUENTA_CATEGORIA = EXTERNAL / "crosswalks/cuenta_categoria.csv"
CW_FUSIONES = EXTERNAL / "crosswalks/fusiones.csv"

errores = []  # acumulador de errores no-fatales para reporte final
def warn(seccion, msg):
    errores.append((seccion, msg))
    print(f"  WARN [{seccion}] {msg}")

## 1. Esquema

In [2]:
ARCHIVOS_ESPERADOS = {
    DIMENSIONES / "dim_cuentas.parquet": ["codigo_cuenta", "denominacion", "fecha_baja", "nivel", "es_regularizadora", "es_cera"],
    DIMENSIONES / "dim_entidades.parquet": ["codigo_entidad", "nombre", "estado", "es_vigente", "es_agrupamiento"],
    DIMENSIONES / "dim_grupos.parquet": ["codigo_entidad", "codigo_grupo"],
    DIMENSIONES / "dim_provincias.parquet": ["iso_codigo", "nombre_bcra"],
    PANELES / "panel_balance_mensual.parquet": ["codigo_entidad", "yyyymm", "fecha", "codigo_cuenta", "saldo"],
    PANELES / "panel_balance_mensual_pre2020.parquet": ["codigo_entidad", "yyyymm", "fecha", "codigo_cuenta", "saldo", "moneda_homogenea"],
    PANELES / "panel_balance_agregados.parquet": ["codigo_entidad", "yyyymm", "codigo_linea", "saldo"],
    PANELES / "panel_distribgeo.parquet": ["codigo_entidad", "yyyymm_corte", "provincia", "prestamos", "depositos"],
    PANELES / "panel_actividad_total.parquet": ["actfec", "actmon"],
    PANELES / "panel_actividad_grupo.parquet": ["actfec", "actmon"],
    PANELES / "panel_actividad_localidad.parquet": ["locprovi", "locfec", "locmon"],
    PANELES / "panel_indicadores.parquet": ["codigo_entidad", "yyyymm", "codigo_linea", "valor"],
    PANELES / "panel_esd.parquet": ["codigo_entidad", "yyyymm", "codigo_linea", "valor"],
    PANELES / "panel_estructura.parquet": ["codigo_entidad", "yyyymm", "codigo_linea", "valor"],
    PANELES / "panel_sucursales_provincia.parquet": ["codigo_entidad", "fecha_corte", "codigo_provincia"],
}

for path, cols_min in ARCHIVOS_ESPERADOS.items():
    assert path.exists(), f"FALTA archivo: {path.relative_to(REPO)}"
    df = pd.read_parquet(path)
    faltantes = [c for c in cols_min if c not in df.columns]
    assert not faltantes, f"En {path.name} faltan columnas: {faltantes}"

print(f"Esquema OK: {len(ARCHIVOS_ESPERADOS)} archivos verificados.")

Esquema OK: 15 archivos verificados.


## 2. Integridad de claves

In [3]:
def chequear_clave(path, cols_clave, nombre):
    df = pd.read_parquet(path)
    n_dup = df[cols_clave].duplicated().sum()
    n_null = df[cols_clave].isna().any(axis=1).sum()
    assert n_dup == 0, f"{nombre}: {n_dup} duplicados en clave {cols_clave}"
    assert n_null == 0, f"{nombre}: {n_null} nulls en clave {cols_clave}"
    print(f"  {nombre:55s}  filas={len(df):>10,}  clave OK")

print("Claves primarias:")
chequear_clave(DIMENSIONES / "dim_cuentas.parquet", ["codigo_cuenta"], "dim_cuentas")
chequear_clave(DIMENSIONES / "dim_entidades.parquet", ["idx"], "dim_entidades")
chequear_clave(DIMENSIONES / "dim_grupos.parquet", ["codigo_entidad", "codigo_grupo"], "dim_grupos")
chequear_clave(DIMENSIONES / "dim_provincias.parquet", ["iso_codigo"], "dim_provincias")
chequear_clave(PANELES / "panel_balance_mensual.parquet", ["codigo_entidad", "yyyymm", "codigo_cuenta"], "panel_balance_mensual")
chequear_clave(PANELES / "panel_balance_mensual_pre2020.parquet", ["codigo_entidad", "yyyymm", "codigo_cuenta"], "panel_balance_mensual_pre2020")
chequear_clave(PANELES / "panel_balance_agregados.parquet", ["codigo_entidad", "yyyymm", "codigo_linea"], "panel_balance_agregados")
chequear_clave(PANELES / "panel_distribgeo.parquet", ["codigo_entidad", "yyyymm_corte", "provincia"], "panel_distribgeo")
chequear_clave(PANELES / "panel_indicadores.parquet", ["codigo_entidad", "yyyymm", "codigo_linea"], "panel_indicadores")
chequear_clave(PANELES / "panel_esd.parquet", ["codigo_entidad", "yyyymm", "codigo_linea"], "panel_esd")
chequear_clave(PANELES / "panel_estructura.parquet", ["codigo_entidad", "yyyymm", "codigo_linea"], "panel_estructura")
chequear_clave(PANELES / "panel_sucursales_provincia.parquet", ["codigo_entidad", "fecha_corte", "codigo_provincia"], "panel_sucursales_provincia")

Claves primarias:
  dim_cuentas                                              filas=     5,519  clave OK
  dim_entidades                                            filas=       312  clave OK
  dim_grupos                                               filas=       275  clave OK
  dim_provincias                                           filas=        25  clave OK
  panel_balance_mensual                                    filas= 1,390,274  clave OK


  panel_balance_mensual_pre2020                            filas= 7,853,701  clave OK
  panel_balance_agregados                                  filas=    86,474  clave OK
  panel_distribgeo                                         filas=    12,874  clave OK
  panel_indicadores                                        filas=   303,795  clave OK
  panel_esd                                                filas=   112,049  clave OK
  panel_estructura                                         filas=   110,581  clave OK
  panel_sucursales_provincia                               filas=    41,916  clave OK


## 3. Cobertura temporal

In [4]:
rangos_esperados = {
    "panel_balance_mensual.parquet": (202001, 202601),
    "panel_balance_mensual_pre2020.parquet": (199411, 201912),
    "panel_balance_agregados.parquet": (None, 202601),  # min variable; max debe llegar al dump
    "panel_distribgeo.parquet": (None, 202512),
    "panel_indicadores.parquet": (None, 202601),
    "panel_esd.parquet": (None, 202512),
    "panel_estructura.parquet": (None, 202601),
}

print("Rangos temporales:")
for nombre, (esperado_min, esperado_max) in rangos_esperados.items():
    df = pd.read_parquet(PANELES / nombre)
    col_fecha = "yyyymm_corte" if "yyyymm_corte" in df.columns else "yyyymm"
    if col_fecha not in df.columns:
        col_fecha = "fecha_corte"
    if df[col_fecha].dtype == "object":
        vmin, vmax = df[col_fecha].min(), df[col_fecha].max()
    else:
        vmin, vmax = int(df[col_fecha].min()), int(df[col_fecha].max())

    if esperado_min is not None:
        assert vmin == esperado_min, f"{nombre}: min={vmin} esperado {esperado_min}"
    if esperado_max is not None:
        assert vmax >= esperado_max, f"{nombre}: max={vmax} debe ser >= {esperado_max}"
    print(f"  {nombre:50s}  rango: {vmin} - {vmax}")

Rangos temporales:
  panel_balance_mensual.parquet                       rango: 202001 - 202601


  panel_balance_mensual_pre2020.parquet               rango: 199411 - 201912
  panel_balance_agregados.parquet                     rango: 201512 - 202601
  panel_distribgeo.parquet                            rango: 201803 - 202512
  panel_indicadores.parquet                           rango: 201512 - 202601
  panel_esd.parquet                                   rango: 201412 - 202512
  panel_estructura.parquet                            rango: 201512 - 202601


## 4. Cobertura de entidades

In [5]:
dim_ent = pd.read_parquet(DIMENSIONES / "dim_entidades.parquet")
panel = pd.read_parquet(PANELES / "panel_balance_mensual.parquet", columns=["codigo_entidad"])

# Entidades únicas en el panel principal
ent_panel = set(panel.codigo_entidad.unique())

# Entidades vigentes (deben estar todas en el panel)
ent_vigentes = set(dim_ent[dim_ent.es_vigente & ~dim_ent.es_agrupamiento].codigo_entidad)

# Entidades inactivas pre-2020 (pueden o no estar en panel; si están es porque tuvieron actividad post-2020)
ent_bajas = set(dim_ent[dim_ent.estado == "BAJA"].codigo_entidad)

vigentes_sin_panel = ent_vigentes - ent_panel
panel_sin_dim = ent_panel - set(dim_ent.codigo_entidad.unique())

# Las vigentes tienen que estar todas en el panel (con tolerancia: alguna recién creada puede no haber reportado todavía)
print(f"Entidades en panel: {len(ent_panel)}")
print(f"Vigentes en dim_entidades: {len(ent_vigentes)}")
print(f"Vigentes que NO están en panel: {len(vigentes_sin_panel)}")
if vigentes_sin_panel:
    detalle = dim_ent[dim_ent.codigo_entidad.isin(vigentes_sin_panel)][["codigo_entidad", "nombre"]]
    print(detalle.to_string(index=False))
    if len(vigentes_sin_panel) > 5:
        warn("cobertura_entidades", f"{len(vigentes_sin_panel)} entidades vigentes sin actividad en panel principal")

assert len(panel_sin_dim) == 0, f"Hay {len(panel_sin_dim)} entidades en panel sin entrada en dim_entidades: {panel_sin_dim}"
print("Cobertura de entidades OK")

Entidades en panel: 80
Vigentes en dim_entidades: 75
Vigentes que NO están en panel: 2
codigo_entidad                                       nombre
         00123 Bank of China Limited, Sucursal Buenos Aires
         00515  BANK OF CHINA LIMITED SUCURSAL BUENOS AIRES
Cobertura de entidades OK


## 5. Cuentas: códigos huérfanos

In [6]:
dim_cu = pd.read_parquet(DIMENSIONES / "dim_cuentas.parquet")
codigos_dim = set(dim_cu.codigo_cuenta)

panel_cu = set(pd.read_parquet(PANELES / "panel_balance_mensual.parquet", columns=["codigo_cuenta"]).codigo_cuenta.unique())
huerfanos = panel_cu - codigos_dim

print(f"Cuentas en panel principal: {len(panel_cu)}")
print(f"Cuentas en dim_cuentas: {len(codigos_dim)}")
print(f"Cuentas en panel sin entrada en dim_cuentas: {len(huerfanos)}")
assert len(huerfanos) == 0, f"Hay códigos huérfanos: {sorted(huerfanos)[:10]}"
print("Cuentas OK: ningún código huérfano")

Cuentas en panel principal: 1454
Cuentas en dim_cuentas: 5519
Cuentas en panel sin entrada en dim_cuentas: 0
Cuentas OK: ningún código huérfano


## 6. Spot checks de eventos conocidos

### 6.1. Cuentas CERA con saldo > 0 a partir de julio 2024

La Com. "A" 8062 (15-jul-2024) creó la CERA. La Com. "A" 8071 (23-jul-2024) dio de alta las cuatro cuentas en el plan contable. Por lo tanto, esperamos saldo > 0 en al menos algún banco a partir de jul-2024 (ago-2024 ya con seguridad).

In [7]:
# IMPORTANTE: el balance BCRA usa convención contable. Activos positivos, pasivos negativos.
# Las cuentas CERA son pasivos (depósitos) -> saldos negativos. Usamos abs() o != 0.
saldos_cera = duckdb.sql(f"""
    select yyyymm, count(distinct codigo_entidad) as bancos_con_saldo,
           abs(sum(saldo)) / 1e9 as saldo_total_miles_millones
    from '{PANELES / "panel_balance_mensual.parquet"}'
    where codigo_cuenta in ('311793', '312183', '315794', '316147')
      and saldo != 0
    group by yyyymm
    order by yyyymm
""").df()

saldos_post_jul24 = saldos_cera[saldos_cera.yyyymm >= 202407]
saldos_pre_jul24 = saldos_cera[saldos_cera.yyyymm < 202407]

assert len(saldos_post_jul24) > 0, "Las cuentas CERA no tienen saldo desde jul-2024 — algo muy mal"
assert saldos_post_jul24.bancos_con_saldo.max() > 5, "Pocos bancos con CERA — el blanqueo debería haber tenido alcance amplio"
assert len(saldos_pre_jul24) == 0, f"Hay saldo CERA antes de jul-2024 — imposible: {saldos_pre_jul24.to_dict()}"

print("CERA spot check OK (recordar: pasivos en BCRA son negativos por convención contable)")
print(saldos_cera.to_string(index=False))

CERA spot check OK (recordar: pasivos en BCRA son negativos por convención contable)
 yyyymm  bancos_con_saldo  saldo_total_miles_millones
 202407                 4                    1.708377
 202408                26                  540.398108
 202409                42                11740.709637
 202410                44                12182.642843
 202411                39                 7842.834884
 202412                38                 6021.710483
 202501                38                 5297.743748
 202502                38                 4783.897778
 202503                38                 4292.875765
 202504                38                 4190.586139
 202505                38                 3896.282107
 202506                37                 3609.020801
 202507                37                 3791.622689
 202508                37                 3497.253918
 202509                37                 3374.768042
 202510                37                 3376.9090

### 6.2. Blanqueo Macri 2016-17 (Ley 27.260) en panel pre-2020

In [8]:
saldos_macri = duckdb.sql(f"""
    select yyyymm, count(distinct codigo_entidad) as bancos_con_saldo,
           abs(sum(saldo)) / 1e9 as saldo_total_miles_millones
    from '{PANELES / "panel_balance_mensual_pre2020.parquet"}'
    where codigo_cuenta in ('311781', '311782', '311783', '315781', '315782', '315783')
      and saldo != 0
    group by yyyymm
    order by yyyymm
""").df()

assert len(saldos_macri) > 0, "El blanqueo Macri no se ve en pre-2020 — verificar codigos de cuenta"
print(f"Blanqueo Macri spot check OK: {len(saldos_macri)} meses con saldo")
print(saldos_macri.head(10).to_string(index=False))

Blanqueo Macri spot check OK: 41 meses con saldo
 yyyymm  bancos_con_saldo  saldo_total_miles_millones
 201608                13                    0.129889
 201609                24                    2.594122
 201610                48                   47.652769
 201611                50                  108.954463
 201612                50                   96.434972
 201701                50                   95.201624
 201702                50                   91.699897
 201703                49                   86.815789
 201704                49                   72.587273
 201705                48                   41.240152


## 7. Consistencia agregado vs individuales

Para una serie reportada tanto a nivel sistema (en `panel_balance_agregados`, vía `balres/AA000`) como a nivel banco (en `panel_balance_mensual`), la suma de los bancos debería coincidir con el agregado.

Caveat técnico: `panel_balance_agregados` viene de `balres/` (balance resumido, ~50 capítulos agregados), mientras que `panel_balance_mensual` viene de `bal_hist/` (plan completo, 5,519 cuentas). El cross-check directo a nivel cuenta requiere mapear las cuentas del plan a los códigos de capítulo de `balres`. Acá hacemos un check más laxo: la cantidad de entidades reportando por mes en cada panel debe ser razonable y consistente.

In [9]:
ent_mes_principal = duckdb.sql(f"""
    select yyyymm, count(distinct codigo_entidad) as n_entidades
    from '{PANELES / "panel_balance_mensual.parquet"}'
    group by yyyymm
    order by yyyymm
""").df()

ent_mes_agregado = duckdb.sql(f"""
    select yyyymm, count(distinct codigo_entidad) as n_grupos
    from '{PANELES / "panel_balance_agregados.parquet"}'
    group by yyyymm
    order by yyyymm
""").df()

print("Cantidad de entidades reportando por mes (panel principal, últimos 6 meses):")
print(ent_mes_principal.tail(6).to_string(index=False))
print()
print("Cantidad de agrupamientos reportando por mes (panel agregados, últimos 6 meses):")
print(ent_mes_agregado.tail(6).to_string(index=False))

# Aserciones laxas
assert ent_mes_principal.n_entidades.median() >= 60, "Pocos bancos reportando por mes — debería haber ~70-80"
assert ent_mes_agregado.n_grupos.median() >= 5, "Pocos agrupamientos por mes"
print()
print("Consistencia entidades-agrupamientos OK (chequeo laxo)")

Cantidad de entidades reportando por mes (panel principal, últimos 6 meses):
 yyyymm  n_entidades
 202508           73
 202509           73
 202510           73
 202511           73
 202512           73
 202601           73

Cantidad de agrupamientos reportando por mes (panel agregados, últimos 6 meses):
 yyyymm  n_grupos
 202508        11
 202509        11
 202510        11
 202511        11
 202512        11
 202601        11

Consistencia entidades-agrupamientos OK (chequeo laxo)


## 8. Crosswalk cuenta_categoria — todos los códigos existen

Para los códigos exactos del crosswalk (no patterns), verifico que existan en `dim_cuentas`. Para los patterns con `%`, verifico que hagan match con al menos una cuenta.

In [10]:
cw = pd.read_csv(CW_CUENTA_CATEGORIA, dtype=str)
print(f"Crosswalk: {len(cw)} filas, {cw.categoria.nunique()} categorías")

dim_cu_set = set(dim_cu.codigo_cuenta)

problemas = []
for _, row in cw.iterrows():
    pat = row["codigo_cuenta_pattern"]
    if "%" in pat:
        # Pattern: convertir a regex y matchear
        regex_pat = "^" + pat.replace("%", ".*") + "$"
        matches = dim_cu[dim_cu.codigo_cuenta.str.match(regex_pat)]
        if len(matches) == 0:
            problemas.append((row.categoria, pat, "ningún match"))
    else:
        if pat not in dim_cu_set:
            problemas.append((row.categoria, pat, "código no existe en dim_cuentas"))

if problemas:
    for p in problemas:
        warn("crosswalk_cuenta_categoria", f"{p[0]} | {p[1]} | {p[2]}")
    raise AssertionError(f"{len(problemas)} problemas en crosswalk")

print("Crosswalk OK: todos los códigos y patterns matchean algo en dim_cuentas")

Crosswalk: 76 filas, 30 categorías
Crosswalk OK: todos los códigos y patterns matchean algo en dim_cuentas


## 9. Cross-check del crosswalk: agregación reproduce CERA conocida

Aplico el crosswalk al panel principal y verifico que la categoría `cera_total` reproduce los números esperados (suma de las 4 cuentas explícitas en jul-2024 en adelante).

In [11]:
cera_via_crosswalk = duckdb.sql(f"""
    select p.yyyymm, sum(p.saldo) / 1e12 as saldo_billones_pesos
    from '{PANELES / "panel_balance_mensual.parquet"}' p
    join '{CW_CUENTA_CATEGORIA}' cw
      on p.codigo_cuenta like cw.codigo_cuenta_pattern
    where cw.categoria = 'cera_total'
    group by p.yyyymm
    having yyyymm >= 202407
    order by yyyymm
""").df()

cera_directo = duckdb.sql(f"""
    select yyyymm, sum(saldo) / 1e12 as saldo_billones_pesos
    from '{PANELES / "panel_balance_mensual.parquet"}'
    where codigo_cuenta in ('311793', '312183', '315794', '316147')
      and yyyymm >= 202407
    group by yyyymm
    order by yyyymm
""").df()

# Deben ser idénticas
assert cera_via_crosswalk.equals(cera_directo), "El crosswalk no reproduce el saldo CERA directo — revisar"
print("Crosswalk reproduce CERA correctamente. Últimos 6 meses:")
print(cera_via_crosswalk.tail(6).to_string(index=False))

Crosswalk reproduce CERA correctamente. Últimos 6 meses:
 yyyymm  saldo_billones_pesos
 202508             -3.497254
 202509             -3.374768
 202510             -3.376909
 202511             -3.322064
 202512             -3.317077
 202601             -0.863739


## 10. Resumen final

In [12]:
print("="*70)
print("VALIDACION COMPLETA")
print("="*70)
print(f"Errores fatales: 0 (de lo contrario el notebook se hubiera interrumpido)")
print(f"Warnings (no fatales): {len(errores)}")
for s, m in errores:
    print(f"  [{s}] {m}")

if not errores:
    print("\nTodos los chequeos pasaron sin warnings.")

VALIDACION COMPLETA
Errores fatales: 0 (de lo contrario el notebook se hubiera interrumpido)
Warnings (no fatales): 0

Todos los chequeos pasaron sin warnings.
